In [1]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import Point
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker
from matplotlib import pyplot as plt
from config import *
import os
import rasterio
from rasterio.mask import mask

pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [2]:
# Main filepaths
metadata_filepath = "../INPUT/01_metadata/metadata.csv"
basinatlas_filepath = "../INPUT/04_static_basinatlas_catchment_characteristics/static_basinatlas_catchment_characteristics.csv"
COMID_filepath = "../INPUT/00_geospatial_data/SWIM_gage_loc/SWIM_gage_loc.shp"
USSF_filepath = "../INPUT/00_geospatial_data/US_shapefiles/cb_2017_us_state_5m.shp"
INPUT_filepath = "../INPUT/"
OUTPUT_filepath = "../OUTPUT/"
shapefile_filepath = INPUT_filepath+'03_shapefiles'

In [3]:
# Reading and filtering 
basinatlas = pd.read_csv(basinatlas_filepath)
basinatlas = basinatlas[['STREAM_ID',
                         'ari_ix_sav', # Aridity Index (spatial ave)
                         'ppd_pk_sav', # Pop density (pop km2; spatial ave)
                         'rdd_mk_sav', # Road Density (spatial ave)
                         'ire_pc_sse', # Irrigated Area Extent (spatial ext. %)
                         'cly_pc_sav', # Clay percent (SoilGrid data; top 0-5 cm)
                         'slt_pc_sav', # Silt percent (SoilGrid data; top 0-5 cm)
                         'snd_pc_sav', # Sand percent (SoilGrid data; top 0-5 cm)
                         'soc_th_sav', # SOC (SoilGrid data; top 0-5 cm)
                        ]]

param_dict = {'ari_ix_sav':'Aridity Index',
              'kar_pc_sse':'% Karst', 
              'ppd_pk_sav':'Pop. Density',
              'rdd_mk_sav':'Road Density',
              'slp_dg_sav':'Terrain slope',
              'lit_cl_smj_remap':'Lithology classes',
              'ele_mt_sav':'Average elevation'}

# Converting to true values
basinatlas['ari_ix_sav'] = basinatlas['ari_ix_sav']/100
basinatlas = basinatlas[basinatlas['STREAM_ID'].isin(main_sites)]
basinatlas.head(4)

,STREAM_ID,ari_ix_sav,ppd_pk_sav,rdd_mk_sav,ire_pc_sse,cly_pc_sav,slt_pc_sav,snd_pc_sav,soc_th_sav
137,STREAM-gauge-4431,0.942310,101.269928,869.820678,0.000000,24.395669,49.164636,25.835364,27.142266
193,STREAM-gauge-3776,1.054271,27.952290,381.387915,0.048819,23.575847,40.323986,36.129844,30.054426
197,STREAM-gauge-2886,0.900000,14.056000,404.000000,1.000000,23.000000,50.000000,27.000000,22.000000
203,STREAM-gauge-2891,0.920129,20.969770,343.763666,0.919432,24.741307,50.755877,24.571676,20.623302


In [4]:
# Read in watershede shapefiles
# Get all shapefile names (without extension)
shapefile_names = [
    os.path.splitext(f)[0] 
    for f in os.listdir(shapefile_filepath) 
    if f.endswith('.shp')
]

# Check matches
ag_sites = [name for name in shapefile_names if name in main_sites]

# WWTP density
Macedo, H. E., Lehner, B., Nicell, J. A., Grill, G., Li, J., Limtong, A., & Shakya, R. (2021). HydroWASTE version 1.0 [Data set]. https://doi.org/10.6084/m9.figshare.14847786.v1


In [5]:
WWTP_data = pd.read_csv(INPUT_filepath+'00_additional_data/HydroWASTE_v10/HydroWASTE_v10.csv', 
                         encoding='latin-1')
WWTP_data.head()

,WASTE_ID,SOURCE,ORG_ID,WWTP_NAME,COUNTRY,CNTRY_ISO,LAT_WWTP,LON_WWTP,QUAL_LOC,LAT_OUT,LON_OUT,STATUS,POP_SERVED,QUAL_POP,WASTE_DIS,QUAL_WASTE,LEVEL,QUAL_LEVEL,DF,HYRIV_ID,RIVER_DIS,COAST_10KM,COAST_50KM,DESIGN_CAP,QUAL_CAP
0,1,1,1140441,Akmenes aglomeracija,Lithuania,LTU,56.247,22.726,2,56.223,22.627,Not Reported,1060,2,148.213,4,Advanced,1,2421.974,20228874.0,4.153,0,0,4600.0,2
1,2,1,1140443,Alytaus m aglomeracija,Lithuania,LTU,54.432,24.056,2,54.519,24.098,Not Reported,87900,2,8797.904,1,Advanced,1,2534.527,20261585.0,257.983,0,0,220000.0,2
2,3,1,1140445,Anyksciu aglomeracija,Lithuania,LTU,55.509,25.073,2,55.452,25.006,Not Reported,12400,2,1959.285,1,Advanced,1,1367.809,20243105.0,30.995,0,0,33000.0,2
3,4,1,1140447,Ariogalos aglomeracija,Lithuania,LTU,55.252,23.484,2,55.210,23.510,Not Reported,2500,2,578.482,1,Secondary,1,2061.969,20247446.0,13.799,0,0,4357.0,2
4,5,1,1140449,Baisogalos aglomeracija,Lithuania,LTU,55.644,23.741,2,55.681,23.835,Not Reported,1200,2,167.788,4,Secondary,1,209.549,20239330.0,0.405,0,0,1490.0,2


In [6]:
# Create geometry from lat/lon columns (adjust column names as needed)
geometry = [Point(xy) for xy in zip(WWTP_data['LON_WWTP'], WWTP_data['LAT_WWTP'])]
WWTP_gdf = gpd.GeoDataFrame(WWTP_data, geometry=geometry, crs='EPSG:4326')

# Calculate WWTP density for each matching shapefile
wtshd_dens = []

for site in ag_sites:
    # Load the shapefile
    shp_path = f"{shapefile_filepath}/{site}.shp"
    watershed = gpd.read_file(shp_path)
    
    # Reproject to match
    watershed = watershed.to_crs(WWTP_gdf.crs)
    
    # Count WWTPs within the watershed
    wwtp_in_watershed = gpd.sjoin(WWTP_gdf, watershed, predicate='within')
    wwtp_count = len(wwtp_in_watershed)
    
    # Sum POP_SERVED and WASTE_DIS
    pop_served_total = wwtp_in_watershed['POP_SERVED'].sum()
    waste_dis_total = wwtp_in_watershed['WASTE_DIS'].sum()
    
    # Calculate area (reproject to equal-area CRS)
    watershed_projected = watershed.to_crs('EPSG:5070')  # NAD83 Albers for North America
    area_km2 = watershed_projected.geometry.area.sum() / 1e6  # m2 to km2
    
    # Density: WWTPs per 1000 km2
    density = (wwtp_count / area_km2) * 100
    
    wtshd_dens.append({
        'STREAM_ID': site,
        'wwtp_num': wwtp_count,
        'pop_served_total': pop_served_total,
        'waste_dis_total_m3_d': waste_dis_total,
        'area_km2': area_km2,
        'wwtp_density_per_100km2': density
    })

# Create results DataFrame
WWTP_dens = pd.DataFrame(wtshd_dens)
WWTP_dens.head(2)

,STREAM_ID,wwtp_num,pop_served_total,waste_dis_total_m3_d,area_km2,wwtp_density_per_100km2
0,STREAM-gauge-2203,1789,9485598,4462125.993,1.277917e+06,0.139993
1,STREAM-gauge-2804,397,756629,304543.929,1.501125e+05,0.264468


# Tile drainage
Valayamkunnath, P., Barlage, M., Chen, F., Gochis, D. J., & Franz, K. J. (2020). Mapping of 30-meter resolution tile-drained croplands using a geospatial modeling approach. Scientific Data, 7(1), 257.


In [7]:
# reading in tif file
tiledrain_raster_filepath = INPUT_filepath+'00_additional_data/tile_drainage/AgTile-US/TIFF/AgTile-US.tif'

wtshd_td = []

for site in ag_sites:
    shp_path = f"{shapefile_filepath}/{site}.shp"
    watershed = gpd.read_file(shp_path)
    
    with rasterio.open(tiledrain_raster_filepath) as src:
        # Reproject watershed to raster CRS
        watershed_reproj = watershed.to_crs(src.crs)
        
        # Mask raster to watershed boundary
        out_image, out_transform = mask(src, watershed_reproj.geometry, crop=True)
        
        # Get the data (first band)
        data = out_image[0]
        
        # Exclude nodata values (adjust nodata value as needed)
        nodata = src.nodata if src.nodata is not None else -999
        valid_data = data[data != nodata]
        
        # Calculate mean (fraction tile-drained) * 100 = percent
        pct_tile_drained = np.mean(valid_data) * 100
    
    wtshd_td.append({
        'STREAM_ID': site,
        'pct_tile_drained': pct_tile_drained
    })

tiledrain_pct = pd.DataFrame(wtshd_td)
tiledrain_pct.to_csv(INPUT_filepath + '00_additional_data/tile_drainage/watershed_tile_drainage_pct.csv',
                    index=False)

# Irrigation (2017)
Xie, Y., Gibbs, H. K., & Lark, T. J. (2021). Landsat-based Irrigation Dataset (LANID): 30 m resolution maps of irrigation distribution, frequency, and change for the US, 1997–2017. Earth System Science Data, 13(12), 5689–5710.

In [8]:
# read in tif file
irrigation_raster_filepath = INPUT_filepath+'00_additional_data/Irrigation/lanid2017.tif'

wtshd_irr = []

for site in ag_sites:
    shp_path = f"{shapefile_filepath}/{site}.shp"
    watershed = gpd.read_file(shp_path)
    
    with rasterio.open(irrigation_raster_filepath) as src:
        # Reproject watershed to raster CRS
        watershed_reproj = watershed.to_crs(src.crs)
        
        # Mask raster to watershed boundary
        out_image, out_transform = mask(src, watershed_reproj.geometry, crop=True)
        
        # Get the data (first band)
        data = out_image[0]
        
        # Exclude nodata values (adjust nodata value as needed)
        nodata = src.nodata if src.nodata is not None else -999
        valid_data = data[data != nodata]
        
        # Calculate mean (fraction irrigated) * 100 = percent
        pct_irrigated = np.mean(valid_data) * 100
    
    wtshd_irr.append({
        'STREAM_ID': site,
        'pct_irrigated': pct_irrigated
    })
# Create and save results
irrigated_pct = pd.DataFrame(wtshd_irr)
irrigated_pct.to_csv(INPUT_filepath + '00_additional_data/irrigation/watershed_irrigation_pct_2017.csv',
                    index=False)

In [9]:
# Pre-calculate original areas
septic_tank = gpd.read_file('../INPUT/00_additional_data/septic_tank_density/septic_tank_shapefile.shp')
septic_tank = septic_tank.to_crs('EPSG:5070')
septic_tank['Area_m2_recalc'] = septic_tank.area

shapefile_names = [
    os.path.splitext(f)[0] 
    for f in os.listdir(shapefile_filepath) 
    if f.endswith('.shp')
]

# Check matches
ag_sites = [name for name in shapefile_names if name in main_sites]

#%% Load and prepare septic tank data
septic_tank = gpd.read_file(INPUT_filepath + '00_additional_data/septic_tank_density/septic_tank_shapefile.shp')

# Fix invalid geometries
septic_tank['geometry'] = septic_tank.geometry.buffer(0)

# Reproject to equal-area CRS
septic_tank = septic_tank.to_crs('EPSG:5070')

# Pre-calculate original areas
septic_tank['Area_m2_recalc'] = septic_tank.area

print(f"Loaded {len(septic_tank)} septic tank polygons")

#%% Calculate septic density for each watershed
print(f"Processing {len(ag_sites)} watersheds...")
results = []

for site in ag_sites:
    shp_path = f"{shapefile_filepath}/{site}.shp"
    
    # Check if file exists
    if not os.path.exists(shp_path):
        print(f"Warning: {shp_path} not found, skipping...")
        continue
    
    watershed = gpd.read_file(shp_path)
    
    # Fix invalid geometries
    watershed['geometry'] = watershed.geometry.buffer(0)
    
    # Reproject to equal-area CRS
    watershed_proj = watershed.to_crs('EPSG:5070')
    
    # Calculate watershed area in km2
    area_km2 = watershed_proj.geometry.area.sum() / 1e6
    
    # Clip and calculate area-weighted septic count
    clipped = gpd.clip(septic_tank, watershed_proj.geometry.iloc[0])
    
    if len(clipped) > 0:
        clipped['area_fraction'] = clipped.area / clipped['Area_m2_recalc']
        septic_count = np.sum(clipped['MS_SepCnt'] * clipped['area_fraction'])
        septic_density_km2 = septic_count / area_km2
    else:
        septic_count = 0
        septic_density_km2 = 0
    
    results.append({
        'STREAM_ID': site,
        'area_km2': area_km2,
        'septic_count': septic_count,
        'septic_density_km2': septic_density_km2
    })

# Create and save results
septic_df = pd.DataFrame(results)
septic_df.to_csv(INPUT_filepath + '00_additional_data/septic_tank_density/watershed_septic_tank_density.csv',
                    index=False)

Loaded 237900 septic tank polygons
Processing 28 watersheds...


# Conservation Practices
USDA-NASS. (2026). Census of Agriculture. https://www.nass.usda.gov/AgCensus/index.php

In [10]:
# Load data
cons_practices_filepath = INPUT_filepath+'00_additional_data/conservation_practices/CoA_land_practices_2017_2022.csv'
cons_pract = pd.read_csv(cons_practices_filepath)
cons_pract['Value'] = pd.to_numeric(cons_pract['Value'].str.replace(',', ''), errors='coerce')

# Aggregating 2017 and 2022 data using average value; no data available before 2017
avg_cons_pract = cons_pract.groupby(['State ANSI', 'County ANSI','Data Item'])['Value'].mean().reset_index()

# Creating GEOID so it matches with the County shapefile
avg_cons_pract['GEOID'] = (
    avg_cons_pract['State ANSI'].astype(int).astype(str).str.zfill(2) +
    avg_cons_pract['County ANSI'].astype(int).astype(str).str.zfill(3)
)
avg_cons_pract['GEOID'] = avg_cons_pract['GEOID'].astype(int)
avg_cons_pract['area_km2'] = avg_cons_pract['Value']* 0.00404686 # converting acres to km2

# Dropping acres so it's not confusing
avg_cons_pract.drop(columns='Value', inplace = True)

avg_cons_pract.head(2)

,State ANSI,County ANSI,Data Item,GEOID,area_km2
0,1,1,"PRACTICES, LAND USE, CROPLAND, CONSERVATION TI...",1001,6.800748
1,1,1,"PRACTICES, LAND USE, CROPLAND, CONSERVATION TI...",1001,32.401185


In [11]:
# Load county-scale data
cnty_shapefile = gpd.read_file(INPUT_filepath+'00_additional_data/county_shp/US_CountyShapefile_2017.shp')
cnty_shapefile['GEOID'] = cnty_shapefile['GEOID'].astype(int)
cnty_shapefile.head(2)

,STATEFP,COUNTYFP,COUNTYNS,AFFGEOID,GEOID,NAME,LSAD,ALAND,AWATER,AREA_HA,geometry
0,40,035,01101805,0500000US40035,40035,Craig,06,1971897017,3697828,197546.978,"POLYGON ((50214.011 1529716.936, 50213.974 153..."
1,40,041,01101808,0500000US40041,40041,Delaware,06,1911798952,140253641,205202.047,"POLYGON ((88030.476 1467134.064, 88515.296 146..."


In [12]:
wtsh_cons = []

for site in main_sites:
    site_shapfile_filepath = os.path.join(shapefile_filepath, f"{site}.shp")
    wtshd = gpd.read_file(site_shapfile_filepath)
    wtshd = wtshd.to_crs(cnty_shapefile.crs)
    
    wtshd_area = wtshd.area.sum() / 1000000  # m2 to km2

    try:
        wtshd_county = gpd.clip(cnty_shapefile, wtshd)
    except Exception:
        wtshd_county = gpd.overlay(cnty_shapefile, wtshd, how='intersection')

    wtshd_county['frac_area'] = wtshd_county.area / (wtshd_county['AREA_HA'] * 10000) # m2 / m2

    # Merge brings in 4 rows per GEOID (one per Data Item)
    wtshd_county = wtshd_county[['GEOID', 'frac_area']].merge(avg_cons_pract, on='GEOID', how='inner')

    # Weight each data item separately
    wtshd_county['area_weighted_km2'] = wtshd_county['area_km2'] * wtshd_county['frac_area'] 

    # Group by Data Item so they stay separate
    wtshd_summary = wtshd_county.groupby('Data Item')['area_weighted_km2'].sum().reset_index()
    wtshd_summary['pct_area'] = (wtshd_summary['area_weighted_km2'] / wtshd_area)*100
    wtshd_summary['Site'] = site

    wtsh_cons.append(wtshd_summary)
    
# Combing the results
combined_cons = pd.concat(wtsh_cons, ignore_index=True)

# remapping names so they are more concise
combined_cons.replace({"PRACTICES, LAND USE, CROPLAND, CONSERVATION TILLAGE, (EXCL NO-TILL) - ACRES": "cons_conser_till_pct",
                      "PRACTICES, LAND USE, CROPLAND, CONSERVATION TILLAGE, NO-TILL - ACRES" : "cons_no_till_pct",
                      "PRACTICES, LAND USE, CROPLAND, CONVENTIONAL TILLAGE - ACRES" : "cons_conven_till_pct",
                      "PRACTICES, LAND USE, CROPLAND, COVER CROP PLANTED, (EXCL CRP) - ACRES":"cons_cover_crop_pct"}, inplace=True)

combined_cons.to_csv(INPUT_filepath + '00_additional_data/conservation_practices/watershed_cons_practices_pct.csv',
                    index=False)

TopologyException: side location conflict at -597951.11410449061 1885216.2427336152. This can occur if the input geometry is invalid.
TopologyException: Input geom 1 is invalid: Self-intersection at -597861.49820836028 1885209.2198749897
TopologyException: Input geom 1 is invalid: Self-intersection at -597861.49820836028 1885209.2198749897


# Livestock Density
Byrnes (2026), unpublished.

USDA-NASS. (2017). Census of Agriculture. https://www.nass.usda.gov/AgCensus/index.php

In [13]:
# Read in livestock density files
livestock_inv = pd.read_csv(INPUT_filepath+'00_additional_data/livestock_density/livestock_inventory.csv')

In [14]:
# Converting from livestock numbers to animal units (AU)
livestock_inv['AU'] = livestock_inv['Value'] / livestock_inv['ID'].map(au)
AU_density = livestock_inv.groupby(['YEAR', 'GEOID'])['AU'].sum().reset_index()

In [15]:
site = "STREAM-gauge-4442"

site_shapfile_filepath = os.path.join(shapefile_filepath, f"{site}.shp")
wtshd = gpd.read_file(site_shapfile_filepath)
    
# Making sure the crs matches + and that it's in equal-area CRS
wtshd = wtshd.to_crs(cnty_shapefile.crs)

# watershed area
wtshd_area = wtshd.area.sum()/1000000 # km2

# Clipping county to watershed
try:
    wtshd_county = gpd.clip(cnty_shapefile, wtshd)
except Exception:
    wtshd_county = gpd.overlay(cnty_shapefile, wtshd, how='intersection')

# Calculate the fraction of the county contained within the watershed
wtshd_county['frac_area'] = wtshd_county.area / (wtshd_county['AREA_HA'] * 10000)

AU_density['GEOID'].isin(wtshd_county['GEOID'])
wtshd_county = wtshd_county[['GEOID', 'frac_area']].merge(AU_density, on='GEOID', how='inner')

# Area-weighted AU, then sum by year
wtshd_county['AU_weighted'] = wtshd_county['AU'] * wtshd_county['frac_area']
wtshd_year = wtshd_county.groupby('YEAR')['AU_weighted'].sum().reset_index()
wtshd_year['AU_density'] = wtshd_year['AU_weighted']/(wtshd_area)
wtshd_year['Site'] = site

In [16]:
wtshd_year

,YEAR,AU_weighted,AU_density,Site
0,1930,709.010617,33.513350,STREAM-gauge-4442
1,1931,731.243235,34.564236,STREAM-gauge-4442
2,1932,753.475853,35.615122,STREAM-gauge-4442
3,1933,775.708470,36.666008,STREAM-gauge-4442
4,1934,797.941088,37.716895,STREAM-gauge-4442
5,1935,820.423185,38.779573,STREAM-gauge-4442
6,1936,807.537778,38.170509,STREAM-gauge-4442
7,1937,794.652372,37.561444,STREAM-gauge-4442
8,1938,781.766966,36.952380,STREAM-gauge-4442
9,1939,768.881560,36.343316,STREAM-gauge-4442


In [17]:
# Area weighting of livestock density
wtsh_AU = []

for site in main_sites:
    site_shapfile_filepath = os.path.join(shapefile_filepath, f"{site}.shp")
    wtshd = gpd.read_file(site_shapfile_filepath)
        
    # Making sure the crs matches + and that it's in equal-area CRS
    wtshd = wtshd.to_crs(cnty_shapefile.crs)
    
    # watershed area
    wtshd_area = wtshd.area.sum()/1000000 # km2
    
    # Clipping county to watershed
    try:
        wtshd_county = gpd.clip(cnty_shapefile, wtshd)
    except Exception:
        wtshd_county = gpd.overlay(cnty_shapefile, wtshd, how='intersection')

    # Calculate the fraction of the county contained within the watershed
    wtshd_county['frac_area'] = wtshd_county.area / (wtshd_county['AREA_HA'] * 10000)

    AU_density['GEOID'].isin(wtshd_county['GEOID'])
    wtshd_county = wtshd_county[['GEOID', 'frac_area']].merge(AU_density, on='GEOID', how='inner')

    # Area-weighted AU, then sum by year
    wtshd_county['AU_weighted'] = wtshd_county['AU'] * wtshd_county['frac_area']
    wtshd_year = wtshd_county.groupby('YEAR')['AU_weighted'].sum().reset_index()
    wtshd_year['AU_density'] = wtshd_year['AU_weighted']/(wtshd_area)
    wtshd_year['Site'] = site
    
    wtsh_AU.append(wtshd_year)
    
combined_AU = pd.concat(wtsh_AU, ignore_index=True)

combined_AU.to_csv(INPUT_filepath+'00_additional_data/livestock_density/AU_density.csv',
                    index=False)

TopologyException: side location conflict at -597951.11410449061 1885216.2427336152. This can occur if the input geometry is invalid.
TopologyException: Input geom 1 is invalid: Self-intersection at -597861.49820836028 1885209.2198749897
TopologyException: Input geom 1 is invalid: Self-intersection at -597861.49820836028 1885209.2198749897


# Polaris soils

In [18]:
# C:\Users\danyk\Work\11_STREAM_seasonality\INPUT\00_additional_data\polaris_soils
ksat_soils = pd.read_csv(INPUT_filepath+'00_additional_data/polaris_soils/MRB_POLARIS_30_watershed_ksat_weighted.csv')
clay_soils = pd.read_csv(INPUT_filepath+'00_additional_data/polaris_soils/MRB_POLARIS_30_watershed_clay_weighted.csv')
silt_soils = pd.read_csv(INPUT_filepath+'00_additional_data/polaris_soils/MRB_POLARIS_30_watershed_silt_weighted.csv')
sand_soils = pd.read_csv(INPUT_filepath+'00_additional_data/polaris_soils/MRB_POLARIS_30_watershed_sand_weighted.csv')
om_soils = pd.read_csv(INPUT_filepath+'00_additional_data/polaris_soils/MRB_POLARIS_30_watershed_om_weighted.csv')

soil_df = pd.merge(ksat_soils, clay_soils)
soil_df = soil_df.merge(silt_soils)
soil_df = soil_df.merge(sand_soils)
soil_df = soil_df.merge(om_soils)
soil_df.shape

(28, 6)

In [19]:
# BasinAtlas variables (already has STREAM_ID)
combined_df = basinatlas.copy()

# WWTP density
combined_df = combined_df.merge(
    WWTP_dens[['STREAM_ID', 'wwtp_num', 'pop_served_total', 'waste_dis_total_m3_d', 
               'area_km2', 'wwtp_density_per_100km2']],
    on='STREAM_ID', how='outer'
)

# Tile drainage
combined_df = combined_df.merge(
    tiledrain_pct[['STREAM_ID', 'pct_tile_drained']],
    on='STREAM_ID', how='outer'
)

# Irrigation
combined_df = combined_df.merge(
    irrigated_pct[['STREAM_ID', 'pct_irrigated']],
    on='STREAM_ID', how='outer'
)

# Septic tank density
combined_df = combined_df.merge(
    septic_df[['STREAM_ID', 'septic_count', 'septic_density_km2']],
    on='STREAM_ID', how='outer'
)

# Pivot conservation practices from long to wide, then merge
cons_wide = combined_cons.pivot_table(
    index='Site', columns='Data Item', values='pct_area'
).reset_index().rename(columns={'Site': 'STREAM_ID'})

combined_df = combined_df.merge(cons_wide, on='STREAM_ID', how='outer')

# Livestock AU density (Mean AU density across all census years)
combined_AU = combined_AU[combined_AU['YEAR'] >= 2012]
AU_mean = combined_AU.groupby('Site')['AU_density'].mean().reset_index()
AU_mean.columns = ['STREAM_ID', 'AU_density_mean']

combined_df = combined_df.merge(AU_mean, on='STREAM_ID', how='outer')

# POLARIS soils
soil_df_renamed = soil_df.rename(columns={soil_df.columns[0]: 'STREAM_ID'})  # adjust if first col isn't STREAM_ID
combined_df = combined_df.merge(soil_df_renamed, on='STREAM_ID', how='outer')

# Summary
print(f"Final DataFrame shape: {combined_df.shape}")
print(f"Sites: {combined_df['STREAM_ID'].nunique()}")
print(f"\nColumns:\n{list(combined_df.columns)}")

combined_df.head()
combined_df.to_csv(OUTPUT_filepath+'MRB_STREAM_data_basin_characteristics.csv',
                    index=False)

Final DataFrame shape: (28, 28)
Sites: 28

Columns:
['STREAM_ID', 'ari_ix_sav', 'ppd_pk_sav', 'rdd_mk_sav', 'ire_pc_sse', 'cly_pc_sav', 'slt_pc_sav', 'snd_pc_sav', 'soc_th_sav', 'wwtp_num', 'pop_served_total', 'waste_dis_total_m3_d', 'area_km2', 'wwtp_density_per_100km2', 'pct_tile_drained', 'pct_irrigated', 'septic_count', 'septic_density_km2', 'cons_conser_till_pct', 'cons_conven_till_pct', 'cons_cover_crop_pct', 'cons_no_till_pct', 'AU_density_mean', 'kSat_log10_cmhr', 'clay_pct', 'silt_pct', 'sand_pct', 'om_log10_pct']
